In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 07 — LLM Insights

Generate a textual interpretation of the latest saved LSTM forecast using the saved Top 10 news from `06_news_retrieval`.

**Model:** `gemini-3.5-flash-lite` (single model; no automatic fallback).

**Important**
- This notebook does not rerun news retrieval or change the LSTM forecast.
- The saved news article content is used as input. URL Context was tested, but some Yahoo Finance URLs were blocked from automated retrieval, so URLs alone are not a reliable substitute for the saved article content.
- Run the two API-call cells deliberately. Each successful call may consume API quota.
- The two batches are used to keep each request smaller. Their news and risk sections are combined in a review draft; the draft is not automatically treated as the final approved report.

## 1. Imports, environment, and paths

In [2]:
# Load the project API key without printing its value.
env_path = Path("../data/env/.env")
load_dotenv(env_path)

api_key = os.getenv("GEMINI_API_KEY")
assert api_key, f"GEMINI_API_KEY was not found. Check: {env_path.resolve()}"

client = genai.Client(api_key=api_key)

MODEL_ID = "gemini-3.5-flash-lite"
output_dir = Path("../data/llm")
output_dir.mkdir(parents=True, exist_ok=True)

print("Gemini client configured.")
print("Model:", MODEL_ID)
print("API key loaded:", bool(api_key))

Gemini client configured.
Model: gemini-3.5-flash-lite
API key loaded: True


## 2. Load the saved forecast and Top 10 news

In [3]:
prediction_path = Path("../data/lstm/processed/bitcoin_daily_predictions.csv")
news_path = Path("../data/news/processed/bitcoin_news_retrieved_top10.csv")

predictions_df = pd.read_csv(prediction_path)
news_df = pd.read_csv(news_path)

required_prediction_columns = {
    "forecast_date", "prediction_created_at", "predicted_close"
}
required_news_columns = {
    "similarity_rank", "title", "source", "published_at_utc",
    "link", "content", "content_status"
}

assert required_prediction_columns.issubset(predictions_df.columns), (
    f"Missing forecast columns: {required_prediction_columns - set(predictions_df.columns)}"
)
assert required_news_columns.issubset(news_df.columns), (
    f"Missing news columns: {required_news_columns - set(news_df.columns)}"
)

news_df = news_df.sort_values("similarity_rank").copy()
assert len(news_df) == 10, f"Expected 10 saved news articles; found {len(news_df)}"
assert news_df["link"].notna().all(), "At least one news URL is missing."

latest_prediction = predictions_df.sort_values("forecast_date").iloc[-1]

forecast_context = f"""
Forecast date: {latest_prediction['forecast_date']}
Prediction created at: {latest_prediction['prediction_created_at']}
Predicted Bitcoin closing price: {float(latest_prediction['predicted_close']):.2f} USD
Actual closing price: Not available yet
Prediction error: Not available yet
""".strip()

display(predictions_df.sort_values("forecast_date").tail(1))
display(news_df[["similarity_rank", "title", "source", "published_at_utc", "link", "content_status"]])

print("Forecast date:", latest_prediction["forecast_date"])
print("Predicted close:", f"{float(latest_prediction['predicted_close']):,.2f} USD")
print("News articles:", len(news_df))
print("News with saved content:", int(news_df["content"].fillna("").astype(str).str.strip().ne("").sum()))

,forecast_date,prediction_created_at,predicted_close,actual_close,prediction_error,absolute_error
2,2026-10-13,2026-10-13 14:15:19,80441.1875,NaN,NaN,NaN


,similarity_rank,title,source,published_at_utc,link,content_status
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,2026-10-11 19:22:47+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
1,2,"Bitcoin and ethereum prices today, Monday, Oct...",Yahoo Personal Finance,2026-10-12 11:40:18+00:00,https://finance.yahoo.com/personal-finance/inv...,summary_and_content
2,3,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,2026-10-11 12:12:14+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
3,4,"Bitcoin struggles above $83,000 a year after $...",Investing.com,2026-10-11 05:54:26+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
4,5,Bitcoin Ranges Near $83K as Traders Look to US...,decrypt,2026-10-12 11:41:50+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
5,6,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,2026-10-10 17:34:25+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
6,7,"Why Bitcoin Can Fall Before a Fed Rate Hike, T...",MoneyLion,2026-10-12 17:00:27+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
7,8,Bitcoin’s Price Prediction: Will Global M2 Poi...,24/7 Wall St.,2026-10-11 15:00:21+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
8,9,"Bitcoin Dips to $80,000: How Other Coins Fared...",24/7 Wall St.,2026-10-10 10:00:32+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
9,10,Bitcoin Has Closed Four Straight Weeks Above I...,24/7 Wall St.,2026-10-12 12:00:10+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content


Forecast date: 2026-10-13
Predicted close: 80,441.19 USD
News articles: 10
News with saved content: 10


## 3. Define the output requirements

In [4]:
system_prompt = """
You are a financial news interpretation assistant for a Bitcoin forecasting system.

Interpret the supplied LSTM forecast using the saved retrieved news. Do not generate,
modify, or recalculate the numerical forecast.

Rules:
1. Treat the LSTM forecast as the primary quantitative result.
2. Use news as market context; do not claim that news directly caused the LSTM prediction.
3. Use only facts supported by the supplied forecast and saved news content.
4. Do not invent events, prices, statistics, sources, or explanations.
5. Distinguish reported facts from interpretations and potential risks.
6. State uncertainty when evidence is insufficient or conflicting.
7. Do not invent a numerical confidence score or calibrated probability.
8. Preserve source names and URLs when referring to news.
9. Do not present the output as guaranteed returns or financial advice.
10. If an article's saved content is short or incomplete, do not infer missing details from its title or URL.

Generate exactly these four sections:
1. Forecasting Result — forecast date and predicted closing price; make clear it is a prediction, not actual price.
2. Confidence — explain uncertainty in words, without a numerical score.
3. Key Market News — summarize relevant retrieved news and cite source names and URLs.
4. Potential Risks — identify supported risks, separate facts from possible implications, and state evidence limitations.

Write clearly and concisely in Markdown.
""".strip()

print("System prompt ready:", bool(system_prompt))
print("System prompt characters:", len(system_prompt))

System prompt ready: True
System prompt characters: 1458


## 4. Prepare two smaller requests

In [5]:
def make_news_context(batch_df):
    records = []
    for _, row in batch_df.iterrows():
        content = str(row["content"]) if pd.notna(row["content"]) else ""
        records.append(
            f"News {int(row['similarity_rank'])}\n"
            f"Title: {row['title']}\n"
            f"Source: {row['source']}\n"
            f"Published at (UTC): {row['published_at_utc']}\n"
            f"Content status: {row['content_status']}\n"
            f"Saved article content:\n{content}\n"
            f"Source URL: {row['link']}"
        )
    return "\n\n".join(records)

batch1_df = news_df.iloc[:5].copy()
batch2_df = news_df.iloc[5:].copy()

def make_user_prompt(batch_df, batch_label):
    return f"""
Interpret the following Bitcoin forecast using only the supplied forecast and saved news.

LSTM forecast:
{forecast_context}

Retrieved news ({batch_label}):
{make_news_context(batch_df)}

Do not invent facts. Do not claim that news directly caused the LSTM forecast.
Include the source name and URL for each news item discussed.
Return all four required sections.
""".strip()

batch1_user_prompt = make_user_prompt(batch1_df, "Batch 1, similarity ranks 1–5")
batch2_user_prompt = make_user_prompt(batch2_df, "Batch 2, similarity ranks 6–10")

for label, batch_df, prompt in [
    ("Batch 1", batch1_df, batch1_user_prompt),
    ("Batch 2", batch2_df, batch2_user_prompt),
]:
    print(f"{label}: ranks {batch_df['similarity_rank'].tolist()}")
    print(f"  Articles: {len(batch_df)}")
    print(f"  Prompt characters: {len(prompt):,}")
print("API called: No")

Batch 1: ranks [1, 2, 3, 4, 5]
  Articles: 5
  Prompt characters: 23,357
Batch 2: ranks [6, 7, 8, 9, 10]
  Articles: 5
  Prompt characters: 27,732
API called: No


## 5. Generate Batch 1 — run deliberately

In [6]:
# This cell makes one API request.
response_batch1 = client.models.generate_content(
    model=MODEL_ID,
    contents=batch1_user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=2048,
    ),
)

batch1_insight = response_batch1.text

if not batch1_insight:
    raise RuntimeError(
        f"Batch 1 returned no text. Finish reason / candidates: "
        f"{response_batch1.candidates}"
    )

print("Model:", MODEL_ID)
print("Batch 1 generated.")
print("Finish reason:", response_batch1.candidates[0].finish_reason if response_batch1.candidates else "Unknown")
print(batch1_insight)

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"


Model: gemini-3.5-flash-lite
Batch 1 generated.
Finish reason: FinishReason.STOP
### 1. Forecasting Result
* **Forecast date:** 2026-10-13
* **Predicted Bitcoin closing price:** 80,441.19 USD
* **Actual closing price:** Not available yet
* **Prediction error:** Not available yet

*Note: The value above is a quantitative prediction generated by the LSTM forecasting system for the specified date and does not represent an actual recorded price.*

### 2. Confidence
The supplied forecasting data does not provide a numerical confidence score or calibrated probability. Market conditions described in the retrieved news indicate mixed technical and macroeconomic signals. While some indicators—such as Bitcoin recovering above its 50-week moving average and maintaining higher lows—support a longer-term recovery (BeInCrypto, https://finance.yahoo.com/markets/crypto/articles/bitcoin-could-entering-bull-market-192247637.html; 24/7 Wall St., https://finance.yahoo.com/markets/crypto/articles/ai-price-

## 6. Generate Batch 2 — run deliberately

In [7]:
# This cell makes one API request.
response_batch2 = client.models.generate_content(
    model=MODEL_ID,
    contents=batch2_user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=2048,
    ),
)

batch2_insight = response_batch2.text
if not batch2_insight:
    raise RuntimeError(
        f"Batch 2 returned no text. Finish reason / candidates: "
        f"{response_batch2.candidates}"
    )

print("Model:", MODEL_ID)
print("Batch 2 generated.")
print("Finish reason:", response_batch2.candidates[0].finish_reason if response_batch2.candidates else "Unknown")
print(batch2_insight)

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"


Model: gemini-3.5-flash-lite
Batch 2 generated.
Finish reason: FinishReason.STOP
### 1. Forecasting Result
* **Forecast date:** 2026-10-13
* **Prediction created at:** 2026-10-13 14:15:19 UTC
* **Predicted Bitcoin closing price:** 80,441.19 USD
* **Actual closing price:** Not available yet
* **Prediction error:** Not available yet
*(Note: This is a quantitative prediction generated by an LSTM model and does not represent an actual recorded price or guaranteed return.)*

### 2. Confidence
The supplied data and historical market contexts highlight significant uncertainty, meaning the model's output should not be treated as a certainty. Market indicators present conflicting signals: while Bitcoin has recorded four straight weekly closes above its 50-week average (sitting at $76,618) and continues to test break-even support around $80,500, historical precedents for this technical setup diverge sharply—pointing either to a major multi-month rally (as seen in 2023) or a severe correction (as

## 7. Save the two original batch responses

In [8]:
# Run after both API-call cells have succeeded.
assert "batch1_insight" in globals(), "Run the Batch 1 generation cell first."
assert "batch2_insight" in globals(), "Run the Batch 2 generation cell first."

batch1_path = output_dir / "bitcoin_llm_insights_batch1.txt"
batch2_path = output_dir / "bitcoin_llm_insights_batch2.txt"

batch1_path.write_text(batch1_insight.strip() + "\n", encoding="utf-8")
batch2_path.write_text(batch2_insight.strip() + "\n", encoding="utf-8")

print("Batch 1 saved:", batch1_path.resolve())
print("Batch 2 saved:", batch2_path.resolve())
print("Batch 1 characters:", len(batch1_insight))
print("Batch 2 characters:", len(batch2_insight))

Batch 1 saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_batch1.txt
Batch 2 saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_batch2.txt
Batch 1 characters: 5369
Batch 2 characters: 4824


## 8. Validate and extract the four sections

In [9]:
section_names = [
    "Forecasting Result",
    "Confidence",
    "Key Market News",
    "Potential Risks",
]

section_pattern = re.compile(
    r"(?m)^#{0,3}\s*\d*\.?\s*\**"
    r"(Forecasting Result|Confidence|Key Market News|Potential Risks)"
    r"\**\s*:?.*$"
)

def extract_sections(text):
    matches = list(section_pattern.finditer(text))
    sections = {}
    for i, match in enumerate(matches):
        name_match = re.search(
            r"(Forecasting Result|Confidence|Key Market News|Potential Risks)",
            match.group(0),
        )
        name = name_match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[name] = text[start:end].strip(" \n-")
    return sections

sections1 = extract_sections(batch1_insight)
sections2 = extract_sections(batch2_insight)

for label, sections in [("Batch 1", sections1), ("Batch 2", sections2)]:
    missing = [name for name in section_names if not sections.get(name)]
    print(f"{label} sections:", list(sections.keys()))
    print(f"{label} missing sections:", missing)
    if missing:
        print(f"WARNING: Review {label}; required sections are missing or empty.")

print("API called: No")

Batch 1 sections: ['Forecasting Result', 'Confidence', 'Key Market News', 'Potential Risks']
Batch 1 missing sections: ['Confidence']
Batch 2 sections: ['Forecasting Result', 'Confidence', 'Key Market News', 'Potential Risks']
Batch 2 missing sections: ['Confidence']
API called: No


## 9. Build a unified review draft

In [10]:
section_names = [
    "Forecasting Result",
    "Confidence",
    "Key Market News",
    "Potential Risks",
]

section_pattern = re.compile(
    r"(?m)^[ \t]*#{0,3}[ \t]*\d*[.)]?[ \t]*\**[ \t]*"
    r"(Forecasting Result|Confidence|Key Market News|Potential Risks)"
    r"\**[ \t]*:?[ \t]*$"
)

def extract_sections(text):
    matches = list(section_pattern.finditer(text))
    sections = {}

    for i, match in enumerate(matches):
        name = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[name] = text[start:end].strip()

    return sections

sections1 = extract_sections(batch1_insight)
sections2 = extract_sections(batch2_insight)

for label, sections in [("Batch 1", sections1), ("Batch 2", sections2)]:
    missing = [name for name in section_names if not sections.get(name)]
    print(f"{label} missing sections:", missing)
    print(f"{label} section lengths:", {
        name: len(sections.get(name, ""))
        for name in section_names
    })

Batch 1 missing sections: []
Batch 1 section lengths: {'Forecasting Result': 339, 'Confidence': 1154, 'Key Market News': 1696, 'Potential Risks': 2084}
Batch 2 missing sections: []
Batch 2 section lengths: {'Forecasting Result': 366, 'Confidence': 747, 'Key Market News': 2416, 'Potential Risks': 1199}


In [11]:
# Validate both batches.
for label, sections in [("Batch 1", sections1), ("Batch 2", sections2)]:
    missing = [name for name in section_names if not sections.get(name)]
    assert not missing, f"{label} is missing required sections: {missing}"


def remove_generated_sources(text):
    """Remove source lists generated by Gemini; add one unified list later."""
    match = re.search(
        r"(?im)^[ \t]*(?:#{1,6}[ \t]*)?"
        r"\**[ \t]*Sources[ \t]*\**[ \t]*:?[ \t]*$",
        text,
    )
    return text[:match.start()].strip() if match else text.strip()


# Merge news analysis from both batches.
key_market_news = "\n\n".join(
    part for part in [
        remove_generated_sources(sections1["Key Market News"]),
        remove_generated_sources(sections2["Key Market News"]),
    ] if part
)

potential_risks = "\n\n".join(
    part for part in [
        sections1["Potential Risks"].strip(),
        sections2["Potential Risks"].strip(),
    ] if part
)

# Build one unified source list for all 10 articles.
sources_section = "\n\n".join(
    f"News {int(row['similarity_rank'])}: {row['title']}\n"
    f"Source: {row['source']}\n"
    f"Published at (UTC): {row['published_at_utc']}\n"
    f"URL: {row['link']}\n"
    f"Saved content status: {row['content_status']}"
    for _, row in news_df.sort_values("similarity_rank").iterrows()
)

# Build the complete report first.
unified_report = f"""Bitcoin LLM Insights

1. Forecasting Result

{sections1["Forecasting Result"].strip()}

2. Confidence

{sections1["Confidence"].strip()}

3. Key Market News

{key_market_news}

4. Potential Risks

{potential_risks}

Sources

{sources_section}
"""

# Remove Markdown formatting from the entire report.
unified_report = re.sub(r"(?m)^\s*#{1,6}\s*", "", unified_report)
unified_report = unified_report.replace("**", "").replace("*", "")
unified_report = re.sub(r"(?m)^\s*[-_]{3,}\s*$", "", unified_report)
unified_report = re.sub(r"\n{3,}", "\n\n", unified_report).strip() + "\n"

# Remove leading and trailing spaces from every line.
unified_report = "\n".join(
    line.strip() for line in unified_report.splitlines()
)
unified_report = re.sub(r"\n{3,}", "\n\n", unified_report).strip() + "\n"

# Save once.
review_path = output_dir / "bitcoin_llm_insights_with_urls.txt"
review_path.write_text(unified_report, encoding="utf-8")

print("Unified review saved:", review_path.resolve())
print("Characters:", len(unified_report))
print("Sources included:", len(news_df))
print("Batch labels present:", "Batch 1" in unified_report or "Batch 2" in unified_report)
print("Markdown bold markers present:", "**" in unified_report)
print("Markdown heading markers present:", bool(re.search(r"(?m)^\s*#", unified_report)))
print("API called: No")

Unified review saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_with_urls.txt
Characters: 11922
Sources included: 10
Batch labels present: False
Markdown bold markers present: False
Markdown heading markers present: False
API called: No


## 10. Check report structure and source URLs

In [12]:
saved_review = review_path.read_text(encoding="utf-8")

required_headings = [
    "1. Forecasting Result",
    "2. Confidence",
    "3. Key Market News",
    "4. Potential Risks",
    "Sources",
]
positions = [saved_review.find(heading) for heading in required_headings]

print("Heading checks:")
for heading, position in zip(required_headings, positions):
    print(f"{heading}: {'FOUND' if position >= 0 else 'MISSING'}")

print(
    "Correct main-section order:",
    all(position >= 0 for position in positions)
    and positions == sorted(positions),
)

missing_urls = [
    str(row["link"])
    for _, row in news_df.iterrows()
    if str(row["link"]) not in saved_review
]

print("News URLs missing:", len(missing_urls))
print("News URLs present:", len(news_df) - len(missing_urls), "/", len(news_df))
if missing_urls:
    print("Missing URLs:")
    for url in missing_urls:
        print("-", url)

print("API called: No")

Heading checks:
1. Forecasting Result: FOUND
2. Confidence: FOUND
3. Key Market News: FOUND
4. Potential Risks: FOUND
Sources: FOUND
Correct main-section order: True
News URLs missing: 0
News URLs present: 10 / 10
API called: No


In [15]:
# Remove the entire Sources section and everything after it.
report_without_sources = re.split(
    r"(?im)^\s*Sources\s*$",
    unified_report,
    maxsplit=1
)[0].strip()

# Save a separate CSV without the Sources section.
csv_without_sources_path = (
    output_dir / "bitcoin_llm_insights_without_urls.csv"
)

pd.DataFrame({
    "llm_insights": [report_without_sources]
}).to_csv(
    csv_without_sources_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV saved:", csv_without_sources_path.resolve())
print("Sources section removed:", "Sources" not in report_without_sources)
print("Report characters:", len(report_without_sources))
print("API called: No")

CSV saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_without_urls.csv
Sources section removed: True
Report characters: 8845
API called: No
